# Custom Segmentation Training

Train a custom per-object segmentation model on your Roboflow dataset (COCO format).

**Important:** Set **Runtime > Change runtime type > GPU** before starting.

**After running the install cell, RESTART the runtime** (Runtime > Restart session), then run all cells from the top.

## 1. Download Dataset from Roboflow

Export your Roboflow project using the **Instance Segmentation** annotation type.

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("your-workspace").project("your-project")
version = project.version(1)
dataset = version.download("coco")

## 2. Install Dependencies

**After this cell finishes, RESTART the runtime (Runtime > Restart session) then run all cells from the top again.**

In [ ]:
try:
    import setuptools
    sv = int(setuptools.__version__.split('.')[0])
    assert 70 <= sv < 80, f"setuptools must be 70<=v<80, got {setuptools.__version__}"
    import pkg_resources
    import numpy as np
    assert np.__version__.startswith("1."), f"numpy must be <2, got {np.__version__}"
    import torch, mmcv, mmengine, mmdet
    import openvino as ov
    import nncf
    from mmcv.ops import MultiScaleDeformableAttention
    assert torch.__version__.startswith("2.3")
    assert mmcv.__version__.startswith("2.2")
    print(f"Already installed:")
    print(f"  setuptools: {setuptools.__version__}")
    print(f"  numpy:    {np.__version__}")
    print(f"  torch:    {torch.__version__}")
    print(f"  mmcv:     {mmcv.__version__}")
    print(f"  mmengine: {mmengine.__version__}")
    print(f"  mmdet:    {mmdet.__version__}")
    print(f"  openvino: {ov.__version__}")
    print(f"  nncf:     {nncf.__version__}")
except (ImportError, AssertionError, ModuleNotFoundError):
    print("Installing (first time, ~2-3 min)...")
    
    # Remove broken system pkg_resources (blocks pip's setuptools on Python 3.12)
    !rm -rf /usr/lib/python3/dist-packages/pkg_resources
    
    # Clean slate
    !pip uninstall -y torch torchvision torchaudio mmcv mmcv-lite mmengine mmdet openxlab --quiet
    
    # Torch 2.3.0 + cu121
    !pip install torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu121 --quiet
    
    # mmcv 2.2.0 (only combo with Python 3.12 prebuilt wheels)
    !pip install mmcv==2.2.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.3/index.html --quiet
    
    # mmdet 3.3.0
    !pip install mmengine mmdet==3.3.0 --quiet
    
    # Patch mmdet hardcoded mmcv version check
    !sed -i "s/mmcv_maximum_version = '2.2.0'/mmcv_maximum_version = '2.3.0'/" /usr/local/lib/python3.12/dist-packages/mmdet/__init__.py
    
    # OpenVINO + NNCF + ONNX export
    !pip install openvino nncf onnx onnxruntime --quiet
    
    # CRITICAL: force these LAST so nothing downgrades them
    !pip uninstall -y openxlab --quiet
    !pip install --force-reinstall "setuptools>=70,<80" --quiet
    !pip install "numpy<2" --quiet
    
    print("\n>>> RESTART RUNTIME and run all cells from the top <<<")

## 3. Training Parameters

Adjust `MODEL_SIZE`, `EPOCHS`, `OPTIMIZE`, and `AUG` as needed.

In [ ]:
import os
import json
import shutil
import pickle
import glob
import numpy as np
import torch
from pathlib import Path
from google.colab import files

# ============================================================
# EDIT THIS SECTION
# ============================================================
MODEL_SIZE = "tiny"        # "tiny" | "small" | "medium" | "large"
EPOCHS = 150
BATCH_SIZE = 16
IMAGE_SIZE = 256           # training resolution (tuned for ROI crops from the detection stage)
LR = 0.004                 # base LR (per default 16-batch)

# OPTIMIZE = True   -> produces a smaller, faster model (recommended for edge deployment)
#                      Falls back automatically if the optimization is unstable.
# OPTIMIZE = False  -> skip optimization, keep the standard model.
OPTIMIZE = True

# ============================================================
# AUGMENTATION — training-time only, no inference-speed impact.
#   Mosaic and MixUp are disabled by default because this model is meant to run
#   on ROI crops (one object filling the canvas). Those augmentations create
#   many tiny objects per image, which pulls the model away from the single-object
#   distribution it will see at inference. Turn them on only if you deploy on
#   full scenes rather than crops.
# ============================================================
AUG = {
    "mosaic_prob":       0.0,        # 0.0 disables mosaic (recommended for ROI deployment)
    "mixup_prob":        0.0,        # 0.0 disables mixup (recommended for ROI deployment)
    "random_resize":     (0.5, 2.0), # scale range for RandomResize
    "flip_prob":         0.5,
    "color_photometric": True,       # brightness/contrast/saturation/hue
    "close_mosaic_epochs": 20,       # disable mosaic+mixup for last N epochs (ignored if both are 0)
}

classes = list(project.classes.keys())
project_name = project.name.lower().replace(" ", "_")

print(f"Model size:  {MODEL_SIZE}")
print(f"Classes:     {classes}")
print(f"Epochs:      {EPOCHS}")
print(f"Batch size:  {BATCH_SIZE}")
print(f"Image size:  {IMAGE_SIZE}")
print(f"Optimize:    {OPTIMIZE}")
print(f"Augmentation keys: {list(AUG.keys())}")

## 4. Verify COCO Dataset Layout

Roboflow's COCO instance-segmentation export matches what mmdet expects. This cell sanity-checks that annotations contain polygon masks.

In [ ]:
# Roboflow's COCO instance-segmentation layout:
#   dataset.location/
#     train/_annotations.coco.json + .jpg files
#     valid/_annotations.coco.json + .jpg files
#     test/_annotations.coco.json  + .jpg files
#
# Each annotation has `bbox` and `segmentation` (polygon list).

DATA_ROOT = dataset.location
TRAIN_ANN = f"{DATA_ROOT}/train/_annotations.coco.json"
VAL_ANN   = f"{DATA_ROOT}/valid/_annotations.coco.json"
TEST_ANN  = f"{DATA_ROOT}/test/_annotations.coco.json"

for split, ann_path in [("train", TRAIN_ANN), ("valid", VAL_ANN), ("test", TEST_ANN)]:
    if not os.path.exists(ann_path):
        print(f"  {split}: MISSING  ({ann_path})")
        continue
    with open(ann_path) as f:
        coco = json.load(f)
    img_count = len(coco.get("images", []))
    ann_count = len(coco.get("annotations", []))
    with_mask = sum(1 for a in coco.get("annotations", []) if a.get("segmentation"))
    print(f"  {split}: {img_count} images, {ann_count} annotations ({with_mask} with polygon masks)")

with open(TRAIN_ANN) as f:
    train_coco = json.load(f)
coco_classes = [c["name"] for c in train_coco["categories"] if c["name"] != "__background__"]

print(f"\nClasses from COCO ({len(coco_classes)}): {coco_classes}")

## 5. Build Config & Train

In [ ]:
# Architecture configurations per model size.
# RTMDet-Ins = RTMDet detection head + per-object mask kernel head + shared mask prototype features.
ARCH_CFG = {
    "tiny": {
        "deepen_factor": 0.167,
        "widen_factor":  0.375,
        "exp_on_reg":    False,
        "checkpoint":    "https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet-ins_tiny_8xb32-300e_coco/rtmdet-ins_tiny_8xb32-300e_coco_20221130_151727-ec670f7e.pth",
    },
    "small": {
        "deepen_factor": 0.33,
        "widen_factor":  0.5,
        "exp_on_reg":    False,
        "checkpoint":    "https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet-ins_s_8xb32-300e_coco/rtmdet-ins_s_8xb32-300e_coco_20221121_212604-fdc5d7ec.pth",
    },
    "medium": {
        "deepen_factor": 0.67,
        "widen_factor":  0.75,
        "exp_on_reg":    True,
        "checkpoint":    "https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet-ins_m_8xb32-300e_coco/rtmdet-ins_m_8xb32-300e_coco_20221123_001039-6eba602e.pth",
    },
    "large": {
        "deepen_factor": 1.0,
        "widen_factor":  1.0,
        "exp_on_reg":    True,
        "checkpoint":    "https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet-ins_l_8xb32-300e_coco/rtmdet-ins_l_8xb32-300e_coco_20221124_103237-78d1d652.pth",
    },
}
assert MODEL_SIZE in ARCH_CFG, f"MODEL_SIZE must be one of {list(ARCH_CFG.keys())}"

# Mask head hyperparameters (must match what the inference wrapper expects).
NUM_PROTOTYPES = 8      # channels in shared mask-feature tensor
DYCONV_CHANNELS = 8     # per-detection dynamic conv hidden channels
NUM_DYCONVS = 3         # depth of dynamic conv


def build_detector_config(cfg_arch, coco_classes, data_root, train_ann, val_ann,
                          epochs, batch_size, image_size, lr, aug):
    """Generate training config for custom COCO data with polygon masks."""
    num_classes = len(coco_classes)
    class_tuple = "(" + ", ".join(f'"{c}"' for c in coco_classes) + (",)" if num_classes == 1 else ")")
    palette = "[" + ", ".join("(220, 20, 60)" for _ in coco_classes) + "]"
    stage2_epoch = max(1, epochs - aug["close_mosaic_epochs"])
    eta_min = lr * 0.05

    config = f"""default_scope = 'mmdet'

default_hooks = dict(
    timer=dict(type='IterTimerHook'),
    logger=dict(type='LoggerHook', interval=50),
    param_scheduler=dict(type='ParamSchedulerHook'),
    checkpoint=dict(type='CheckpointHook', interval=10, save_best='coco/segm_mAP', rule='greater', max_keep_ckpts=3),
    sampler_seed=dict(type='DistSamplerSeedHook'),
    visualization=dict(type='DetVisualizationHook'),
)

custom_hooks = [
    dict(type='EMAHook', ema_type='ExpMomentumEMA', momentum=0.0002, update_buffers=True, priority=49),
    dict(
        type='PipelineSwitchHook',
        switch_epoch={stage2_epoch},
        switch_pipeline=[
            dict(type='LoadImageFromFile'),
            dict(type='LoadAnnotations', with_bbox=True, with_mask=True, poly2mask=False),
            dict(type='RandomResize', scale=({image_size}, {image_size}), ratio_range={aug['random_resize']}, keep_ratio=True),
            dict(type='RandomCrop', crop_size=({image_size}, {image_size}), recompute_bbox=True, allow_negative_crop=True),
            dict(type='PhotoMetricDistortion'),
            dict(type='RandomFlip', prob={aug['flip_prob']}),
            dict(type='Pad', size=({image_size}, {image_size}), pad_val=dict(img=(128, 128, 128))),
            dict(type='FilterAnnotations', min_gt_bbox_wh=(1, 1)),
            dict(type='PackDetInputs'),
        ],
    ),
]

env_cfg = dict(
    cudnn_benchmark=False,
    mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0),
    dist_cfg=dict(backend='nccl'),
)

vis_backends = [dict(type='LocalVisBackend')]
visualizer = dict(type='DetLocalVisualizer', vis_backends=vis_backends, name='visualizer')
log_processor = dict(type='LogProcessor', window_size=50, by_epoch=True)
log_level = 'INFO'
load_from = '{cfg_arch['checkpoint']}'
resume = False

# --- Model ---
model = dict(
    type='RTMDet',
    data_preprocessor=dict(
        type='DetDataPreprocessor',
        mean=[103.53, 116.28, 123.675],
        std=[57.375, 57.12, 58.395],
        bgr_to_rgb=False,
        batch_augments=None,
        pad_mask=True,
        mask_pad_value=0,
    ),
    backbone=dict(
        type='CSPNeXt',
        arch='P5',
        expand_ratio=0.5,
        deepen_factor={cfg_arch['deepen_factor']},
        widen_factor={cfg_arch['widen_factor']},
        channel_attention=True,
        norm_cfg=dict(type='SyncBN'),
        act_cfg=dict(type='SiLU', inplace=True),
    ),
    neck=dict(
        type='CSPNeXtPAFPN',
        in_channels=[int(256 * {cfg_arch['widen_factor']}), int(512 * {cfg_arch['widen_factor']}), int(1024 * {cfg_arch['widen_factor']})],
        out_channels=int(256 * {cfg_arch['widen_factor']}),
        num_csp_blocks=max(1, round(3 * {cfg_arch['deepen_factor']})),
        expand_ratio=0.5,
        norm_cfg=dict(type='SyncBN'),
        act_cfg=dict(type='SiLU', inplace=True),
    ),
    bbox_head=dict(
        type='RTMDetInsSepBNHead',
        num_classes={num_classes},
        in_channels=int(256 * {cfg_arch['widen_factor']}),
        stacked_convs=2,
        share_conv=True,
        pred_kernel_size=1,
        feat_channels=int(256 * {cfg_arch['widen_factor']}),
        act_cfg=dict(type='SiLU', inplace=True),
        norm_cfg=dict(type='SyncBN', requires_grad=True),
        anchor_generator=dict(type='MlvlPointGenerator', offset=0, strides=[8, 16, 32]),
        bbox_coder=dict(type='DistancePointBBoxCoder'),
        loss_cls=dict(type='QualityFocalLoss', use_sigmoid=True, beta=2.0, loss_weight=1.0),
        loss_bbox=dict(type='GIoULoss', loss_weight=2.0),
        loss_mask=dict(type='DiceLoss', loss_weight=2.0, eps=5e-06, reduction='mean'),
    ),
    train_cfg=dict(
        assigner=dict(type='DynamicSoftLabelAssigner', topk=13),
        allowed_border=-1,
        pos_weight=-1,
        debug=False,
    ),
    test_cfg=dict(
        nms_pre=1000,
        min_bbox_size=0,
        score_thr=0.05,
        nms=dict(type='nms', iou_threshold=0.6),
        max_per_img=100,
        mask_thr_binary=0.5,
    ),
)

# --- Dataset ---
dataset_type = 'CocoDataset'
data_root = '{data_root}/'
CLASSES = {class_tuple}
metainfo = dict(classes=CLASSES, palette={palette})

backend_args = None

# Stage-1 pipeline with mosaic + mixup.
train_pipeline_stage1 = [
    dict(type='LoadImageFromFile', backend_args=backend_args),
    dict(type='LoadAnnotations', with_bbox=True, with_mask=True, poly2mask=False),
    dict(type='CachedMosaic', img_scale=({image_size}, {image_size}), pad_val=128.0, max_cached_images=40, prob={aug['mosaic_prob']}),
    dict(type='RandomResize', scale=({image_size * 2}, {image_size * 2}), ratio_range={aug['random_resize']}, keep_ratio=True),
    dict(type='RandomCrop', crop_size=({image_size}, {image_size}), recompute_bbox=True, allow_negative_crop=True),
    dict(type='PhotoMetricDistortion'),
    dict(type='RandomFlip', prob={aug['flip_prob']}),
    dict(type='Pad', size=({image_size}, {image_size}), pad_val=dict(img=(128, 128, 128))),
    dict(type='CachedMixUp', img_scale=({image_size}, {image_size}), ratio_range=(1.0, 1.0), max_cached_images=20, pad_val=(128, 128, 128), prob={aug['mixup_prob']}),
    dict(type='FilterAnnotations', min_gt_bbox_wh=(1, 1)),
    dict(type='PackDetInputs'),
]

test_pipeline = [
    dict(type='LoadImageFromFile', backend_args=backend_args),
    dict(type='Resize', scale=({image_size}, {image_size}), keep_ratio=True),
    dict(type='Pad', size=({image_size}, {image_size}), pad_val=dict(img=(128, 128, 128))),
    dict(type='LoadAnnotations', with_bbox=True, with_mask=True, poly2mask=False),
    dict(type='PackDetInputs', meta_keys=('img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor')),
]

train_dataloader = dict(
    batch_size={batch_size},
    num_workers=4,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=True),
    batch_sampler=None,
    dataset=dict(
        type=dataset_type,
        data_root=data_root,
        ann_file='{train_ann}',
        data_prefix=dict(img='train/'),
        filter_cfg=dict(filter_empty_gt=True, min_size=32),
        pipeline=train_pipeline_stage1,
        metainfo=metainfo,
    ),
)

val_dataloader = dict(
    batch_size=1,
    num_workers=2,
    persistent_workers=True,
    drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type=dataset_type,
        data_root=data_root,
        ann_file='{val_ann}',
        data_prefix=dict(img='valid/'),
        test_mode=True,
        pipeline=test_pipeline,
        metainfo=metainfo,
    ),
)

test_dataloader = val_dataloader
val_evaluator = dict(
    type='CocoMetric',
    ann_file=data_root + '{val_ann}',
    metric=['bbox', 'segm'],
    format_only=False,
    backend_args=backend_args,
)
test_evaluator = val_evaluator

train_cfg = dict(type='EpochBasedTrainLoop', max_epochs={epochs}, val_interval=10, dynamic_intervals=[({stage2_epoch}, 1)])
val_cfg = dict(type='ValLoop')
test_cfg = dict(type='TestLoop')

optim_wrapper = dict(
    type='OptimWrapper',
    optimizer=dict(type='AdamW', lr={lr}, weight_decay=0.05),
    paramwise_cfg=dict(norm_decay_mult=0, bias_decay_mult=0, bypass_duplicate=True),
)

param_scheduler = [
    dict(type='LinearLR', start_factor=1e-5, by_epoch=False, begin=0, end=1000),
    dict(type='CosineAnnealingLR', eta_min={eta_min}, begin={epochs // 2}, end={epochs}, T_max={epochs - epochs // 2}, by_epoch=True, convert_to_iter_based=True),
]

auto_scale_lr = dict(base_batch_size=128)
"""
    return config


# ---------- build + train ----------
from mmengine.config import Config
from mmengine.runner import Runner

classes = coco_classes

os.makedirs("/content/configs", exist_ok=True)
config_path = f"/content/configs/segmentation_{MODEL_SIZE}_{project_name}.py"

train_ann_rel = "train/_annotations.coco.json"
val_ann_rel   = "valid/_annotations.coco.json"

config_text = build_detector_config(
    ARCH_CFG[MODEL_SIZE], classes, DATA_ROOT, train_ann_rel, val_ann_rel,
    EPOCHS, BATCH_SIZE, IMAGE_SIZE, LR, AUG,
)
with open(config_path, "w") as f:
    f.write(config_text)
print(f"Config: {config_path}")

work_dir = f"/content/work_dirs/segmentation_{MODEL_SIZE}_{project_name}"
cfg = Config.fromfile(config_path)
cfg.work_dir = work_dir
runner = Runner.from_cfg(cfg)
runner.train()
print(f"\nTraining complete. Work dir: {work_dir}")

## 6. Export & Optimize Model

In [ ]:
import cv2
import openvino as ov
import nncf
from mmdet.apis import init_detector

# Patch torch.onnx.export for compatibility in torch 2.3
import torch.onnx
if not hasattr(torch.onnx, "_dynamo_patched"):
    _orig_export = torch.onnx.export
    def _patched_export(*args, **kwargs):
        kwargs.pop("dynamo", None)
        return _orig_export(*args, **kwargs)
    torch.onnx.export = _patched_export
    torch.onnx._dynamo_patched = True


# ---------- pick best checkpoint ----------
ckpts = sorted(glob.glob(os.path.join(work_dir, "best_*.pth")))
if not ckpts:
    ckpts = sorted(glob.glob(os.path.join(work_dir, "epoch_*.pth")))
assert ckpts, "No checkpoint found!"
checkpoint = ckpts[-1]
print(f"Checkpoint: {checkpoint}")

# Use EMA weights for export (better eval accuracy)
raw_ckpt = torch.load(checkpoint, map_location="cpu")
if "ema_state_dict" in raw_ckpt:
    raw_ckpt["state_dict"] = raw_ckpt["ema_state_dict"]
    print("Using averaged weights")
fixed_ckpt = os.path.join(work_dir, "best_for_export.pth")
torch.save(raw_ckpt, fixed_ckpt)

# ---------- Load model ----------
detector = init_detector(config_path, fixed_ckpt, device="cpu")
detector.eval()


# ---------- Export wrapper: two output tensors ----------
# Output A (detections): (batch, total_anchors, 4 + 1 + num_classes + num_kernel + 3)
#   [cx, cy, w, h,  obj,  cls_probs..,  mask_kernel..,  prior_cx, prior_cy, stride]
# Output B (mask features): (batch, num_prototypes, H/4, W/4) — shared across detections.
#
# At inference time we: filter by obj*max_cls_prob → rotated NMS → for each kept det,
# apply the mask kernel as a tiny dynamic conv on the mask feature map to produce the mask.

PRIOR_OFFSET = 0.0   # MUST match MlvlPointGenerator's `offset` in the training config
NUM_PROTOTYPES = 8
DYCONV_CHANNELS = 8
NUM_DYCONVS = 3
# kernel params per location = (num_proto+2)*C + C + C*C + C + C*1 + 1 = 169 for defaults
NUM_KERNEL_PARAMS = (
    (NUM_PROTOTYPES + 2) * DYCONV_CHANNELS + DYCONV_CHANNELS          # layer 0 weight+bias
    + DYCONV_CHANNELS * DYCONV_CHANNELS + DYCONV_CHANNELS             # layer 1 weight+bias
    + DYCONV_CHANNELS * 1 + 1                                          # layer 2 weight+bias
)


class DetectorExportWrapper(torch.nn.Module):
    def __init__(self, detector, num_classes, strides=(8, 16, 32),
                 prior_offset=PRIOR_OFFSET, num_kernel=NUM_KERNEL_PARAMS):
        super().__init__()
        self.detector = detector
        self.num_classes = num_classes
        self.strides = strides
        self.prior_offset = prior_offset
        self.num_kernel = num_kernel

    def forward(self, x):
        feats = self.detector.extract_feat(x)
        cls_scores, bbox_preds, kernel_preds, mask_feat = self.detector.bbox_head(feats)
        outputs = []
        for cls_s, bbox_p, kern_p, stride in zip(cls_scores, bbox_preds, kernel_preds, self.strides):
            B, _, H, W = cls_s.shape
            device = cls_s.device
            yv, xv = torch.meshgrid(
                torch.arange(H, device=device, dtype=torch.float32),
                torch.arange(W, device=device, dtype=torch.float32),
                indexing='ij',
            )
            grid = torch.stack((xv, yv), dim=-1)
            grid = (grid + self.prior_offset) * stride
            grid = grid.view(1, H * W, 2)
            bbox_p = bbox_p.permute(0, 2, 3, 1).reshape(B, H * W, 4)
            kern_p = kern_p.permute(0, 2, 3, 1).reshape(B, H * W, self.num_kernel)
            x1 = grid[..., 0:1] - bbox_p[..., 0:1]
            y1 = grid[..., 1:2] - bbox_p[..., 1:2]
            x2 = grid[..., 0:1] + bbox_p[..., 2:3]
            y2 = grid[..., 1:2] + bbox_p[..., 3:4]
            xc = (x1 + x2) * 0.5
            yc = (y1 + y2) * 0.5
            w = x2 - x1
            h = y2 - y1
            boxes_xywh = torch.cat([xc, yc, w, h], dim=-1)
            cls_probs = cls_s.permute(0, 2, 3, 1).reshape(B, H * W, self.num_classes).sigmoid()
            obj = cls_probs.max(dim=-1, keepdim=True).values
            stride_t = torch.full((B, H * W, 1), float(stride), device=device)
            prior_info = torch.cat([grid.expand(B, -1, -1), stride_t], dim=-1)  # (B, N, 3)
            outputs.append(torch.cat([boxes_xywh, obj, cls_probs, kern_p, prior_info], dim=-1))
        det_output = torch.cat(outputs, dim=1)
        return det_output, mask_feat


wrapped = DetectorExportWrapper(detector, num_classes=len(classes))
wrapped.eval()

os.makedirs("/content/export", exist_ok=True)
onnx_path = "/content/export/model.onnx"
dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)

torch.onnx.export(
    wrapped, dummy, onnx_path,
    opset_version=17,
    input_names=["input"],
    output_names=["detections", "mask_feat"],
)
print(f"Exported model: {onnx_path} ({os.path.getsize(onnx_path) / 1024 / 1024:.1f} MB)")

# ---------- Standard-precision export ----------
core = ov.Core()
ov_model = core.read_model(onnx_path)
std_xml = "/content/export/model.xml"
ov.save_model(ov_model, std_xml)
std_bin = std_xml.replace(".xml", ".bin")
print(f"Standard model: {std_bin} ({os.path.getsize(std_bin) / 1024 / 1024:.1f} MB)")

# ---------- Optimized model (if requested) ----------
MEAN = np.array([103.53, 116.28, 123.675], dtype=np.float32)
STD  = np.array([57.375, 57.12, 58.395], dtype=np.float32)

def preprocess_for_calibration(img_path, size):
    img = cv2.imread(img_path)
    if img is None:
        return None
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    nh, nw = int(h * r), int(w * r)
    img = cv2.resize(img, (nw, nh))
    padded = np.full((size, size, 3), 128, dtype=np.uint8)
    padded[:nh, :nw] = img
    tensor = ((padded.astype(np.float32) - MEAN) / STD).transpose(2, 0, 1)[None]
    return tensor


opt_xml = None
opt_bin = None

if OPTIMIZE:
    print("\nOptimizing model...")
    calib_imgs = sorted(glob.glob(f"{DATA_ROOT}/train/*.jpg"))[:200]
    calib_tensors = [t for t in (preprocess_for_calibration(p, IMAGE_SIZE) for p in calib_imgs) if t is not None]
    print(f"  Calibration samples: {len(calib_tensors)}")

    ov_model_for_opt = core.read_model(onnx_path)
    calib_dataset = nncf.Dataset(calib_tensors, lambda x: x)
    optimized = nncf.quantize(
        ov_model_for_opt,
        calib_dataset,
        preset=nncf.QuantizationPreset.MIXED,
        subset_size=len(calib_tensors),
    )
    opt_xml_candidate = "/content/export/model_opt.xml"
    ov.save_model(optimized, opt_xml_candidate)
    opt_bin_candidate = opt_xml_candidate.replace(".xml", ".bin")
    print(f"  Optimized model: {opt_bin_candidate} ({os.path.getsize(opt_bin_candidate) / 1024 / 1024:.1f} MB)")
    print(f"  Size reduction: {100 * (1 - os.path.getsize(opt_bin_candidate) / os.path.getsize(std_bin)):.0f}%")

    # Stability check on detection confidence
    compiled_opt = core.compile_model(optimized, "CPU")
    compiled_std = core.compile_model(ov_model, "CPU")
    probe = calib_tensors[0]
    zeros = np.zeros_like(probe)

    def max_detection_score(det_out, num_classes):
        # det layout: [4 box, 1 obj, C cls, K kernel, 3 prior]
        arr = np.asarray(det_out)
        obj = arr[..., 4]
        cls = arr[..., 5:5 + num_classes]
        return float((obj * cls.max(axis=-1)).max())

    nc = len(classes)
    opt_real = list(compiled_opt([probe]).values())
    opt_zero = list(compiled_opt([zeros]).values())
    std_real = list(compiled_std([probe]).values())
    # detections is the first output
    real_opt = max_detection_score(opt_real[0], nc)
    zero_opt = max_detection_score(opt_zero[0], nc)
    real_std = max_detection_score(std_real[0], nc)
    drift = abs(real_opt - real_std) / max(real_std, 1e-8)
    print(f"  Stability check: real_conf={real_opt:.3f}, zero_conf={zero_opt:.3f}, drift={drift*100:.1f}%")

    if real_opt > 0.1 and drift < 0.50:
        opt_xml = opt_xml_candidate
        opt_bin = opt_bin_candidate
        print("  Optimization OK — packaging optimized version")
    else:
        print("  Optimization unstable — packaging standard version only")
else:
    print(f"\nOPTIMIZE={OPTIMIZE} — packaging standard version only.")

## 7. Save & Download Pickle

In [ ]:
# Pickle format mirrors OD/ROD:
#   { bin, xml, cls, colors, meta }
# meta.type is 'iseg' so the inference side can dispatch to the mask path.
# meta also stores the mask-head hyperparameters so the inference wrapper can
# parse the dynamic-conv kernel correctly.

if opt_xml and opt_bin and os.path.exists(opt_xml):
    final_xml_path = opt_xml
    final_bin_path = opt_bin
    precision_tag = "int8"
    variant = "optimized"
else:
    final_xml_path = std_xml
    final_bin_path = std_bin
    precision_tag = "fp32"
    variant = "standard"

with open(final_xml_path, "r", encoding="utf-8") as f:
    xml_data = f.read()
with open(final_bin_path, "rb") as f:
    bin_data = f.read()

model_dict = {
    "bin": bin_data,
    "xml": xml_data,
    "cls": classes,
    "colors": project.colors,
    "meta": {
        "model_type": MODEL_SIZE,
        "type": "iseg",
        "image_size": IMAGE_SIZE,
        "precision": precision_tag,
        "num_prototypes": NUM_PROTOTYPES,
        "dyconv_channels": DYCONV_CHANNELS,
        "num_dyconvs": NUM_DYCONVS,
    },
}

pickle_path = f"/content/{project_name}.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(model_dict, f)

print(f"Pickle contains {variant} model ({len(bin_data) / 1024 / 1024:.1f} MB)")
print(f"\nSaved: {pickle_path}")
print(f"Total size: {os.path.getsize(pickle_path) / 1024 / 1024:.1f} MB")
print(f"Model size: {MODEL_SIZE}")
print(f"Classes: {classes}")
print(f"Image size: {IMAGE_SIZE}")

files.download(pickle_path)